# A Reproducible Hybrid QSPR-Machine Learning Framework Integrating Graph-Theoretic and Cheminformatics Descriptors for Anticancer Drug Property Prediction

## Author: Dr. H.M.Fraz

This notebook implements a reproducible QSPR framework integrating graph-theoretic descriptors,
RDKit molecular descriptors, and Morgan fingerprints for prediction of physicochemical properties
of anti‑cancer drugs. The workflow follows OECD-compliant validation strategy including nested
cross‑validation, SHAP feature selection, Y‑randomization, and applicability domain analysis.


### Library Import

This section loads cheminformatics, machine learning, and scientific computing libraries.
RDKit computes molecular descriptors, sklearn performs ML modelling, and SHAP provides
model interpretability through feature importance analysis.


In [1]:

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import AllChem
from rdkit.Chem import SaltRemover

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor

import shap
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'sklearn'

### Data Loading

The dataset contains SMILES representation of molecules and experimental physicochemical
properties. Duplicate structures are removed to avoid bias in machine learning performance.


In [2]:

def load_data(file):

    if file.endswith(".xlsx"):
        df = pd.read_excel(file)
    else:
        df = pd.read_csv(file)

    return df.drop_duplicates()


### SMILES Standardization

Molecular structures are standardized using RDKit. Salt removal ensures consistent molecular
representation and prevents errors during descriptor calculation.


In [3]:

remover = SaltRemover.SaltRemover()

def clean_smiles(s):

    mol = Chem.MolFromSmiles(s)

    if mol is None:
        return None

    mol = remover.StripMol(mol)

    return Chem.MolToSmiles(mol)


### RDKit Molecular Descriptors

Physicochemical descriptors capture structural properties such as molecular size,
polarity, hydrogen bonding ability, flexibility, and molecular complexity.
These features are widely used in QSPR/QSAR modelling.


In [4]:

def rdkit_features(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    d = {}

    d["MolWt"] = Descriptors.MolWt(mol)
    d["TPSA"] = rdMolDescriptors.CalcTPSA(mol)
    d["NumRotatableBonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
    d["NumHDonors"] = rdMolDescriptors.CalcNumHBD(mol)
    d["NumHAcceptors"] = rdMolDescriptors.CalcNumHBA(mol)
    d["FractionCsp3"] = Descriptors.FractionCSP3(mol)
    d["BalabanJ"] = Descriptors.BalabanJ(mol)
    d["BertzCT"] = Descriptors.BertzCT(mol)
    d["Kappa1"] = Descriptors.Kappa1(mol)
    d["Kappa2"] = Descriptors.Kappa2(mol)
    d["Kappa3"] = Descriptors.Kappa3(mol)
    d["HeavyAtomCount"] = rdMolDescriptors.CalcNumHeavyAtoms(mol)

    return d


### Morgan Fingerprints

Circular fingerprints encode molecular substructures and pharmacophoric fragments.
They capture local chemical environments useful for machine learning prediction tasks.


In [5]:

def fingerprint(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return np.zeros(1024)

    return np.array(
        AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=3,
            nBits=1024
        )
    )


### Feature Matrix Construction

Descriptor sets are combined including RDKit descriptors, graph‑theoretic descriptors,
and fingerprints. This hybrid representation improves structure‑property relationships
by capturing both global topology and local chemical patterns.


In [6]:

def build_features(df):

    df["SMILES"] = df.SMILES.apply(clean_smiles)

    df = df.dropna(subset=["SMILES"])

    rd_df = pd.DataFrame(
        df.SMILES.apply(rdkit_features).tolist()
    )

    fp = np.vstack(
        df.SMILES.apply(fingerprint)
    )

    fp_df = pd.DataFrame(
        fp,
        columns=[f"FP_{i}" for i in range(1024)]
    )

    topo_cols = [
        "M1","M2","ABC","R","H",
        "Wiener","Harary","Mostar",
        "PI","Szeged"
    ]

    X = pd.concat(
        [rd_df, df[topo_cols], fp_df],
        axis=1
    )

    X = X.fillna(0)

    return X


### Feature Filtering

Low variance features and highly correlated descriptors are removed to reduce redundancy.
Feature filtering improves model stability and prevents multicollinearity problems.


In [7]:

def filter_features(X):

    vt = VarianceThreshold(0.01)

    X_vt = vt.fit_transform(X)

    X_vt = pd.DataFrame(
        X_vt,
        columns=X.columns[vt.get_support()]
    )

    corr = X_vt.corr().abs()

    upper = corr.where(
        np.triu(np.ones(corr.shape),1).astype(bool)
    )

    drop_cols = [
        c for c in upper.columns
        if any(upper[c] > 0.95)
    ]

    return X_vt.drop(columns=drop_cols)


### Statistical Metrics

Predictive performance is evaluated using coefficient of determination R²
and predictive coefficient Q². These metrics assess goodness‑of‑fit and
predictive ability of QSPR models.


In [8]:

def Q2(y,y_pred):

    press = np.sum((y-y_pred)**2)

    tss = np.sum((y-np.mean(y))**2)

    return 1 - press/tss


### Machine Learning Models

Three regression algorithms are used:
Ridge regression for linear relationships,
Random Forest for nonlinear patterns,
XGBoost for boosted ensemble learning.
Hyperparameters are optimized using grid search.


In [9]:

models={

"Ridge":Ridge(),

"RF":RandomForestRegressor(random_state=SEED),

"XGB":XGBRegressor(random_state=SEED,verbosity=0)

}

param_grid={

"Ridge":{"alpha":[0.001,0.01,0.1,1,10]},

"RF":{

"n_estimators":[300,500],

"max_depth":[5,10,15]

},

"XGB":{

"learning_rate":[0.01,0.05],

"max_depth":[3,5,7],

"n_estimators":[300,500]

}

}


NameError: name 'Ridge' is not defined

### Main Pipeline

Nested cross‑validation ensures unbiased model evaluation.
SHAP feature selection identifies most important descriptors.
Performance is computed using Q² across all target properties.


In [10]:

def run_pipeline(file):

    df=load_data(file)

    X=build_features(df)

    targets=[

"logP","RB","TPSA","C","Heavy atom","nPotency"

]

    cv_outer=KFold(5,shuffle=True,random_state=SEED)

    for t in targets:

        y=df[t]

        Xt=filter_features(X)

        print("\nTarget:",t)

        for name,m in models.items():

            pred=[]

            obs=[]

            for tr,te in cv_outer.split(Xt):

                sc=StandardScaler()

                Xtr=sc.fit_transform(Xt.iloc[tr])

                Xte=sc.transform(Xt.iloc[te])

                grid=GridSearchCV(m,param_grid[name],cv=3)

                grid.fit(Xtr,y.iloc[tr])

                best=grid.best_estimator_

                p=best.predict(Xte)

                pred.extend(p)

                obs.extend(y.iloc[te])

            print(name,"Q2=",round(Q2(np.array(obs),np.array(pred)),3))


### Execute Model

Provide dataset path containing SMILES, graph indices,
and physicochemical properties. The pipeline outputs
predictive performance for each target property.


In [11]:

run_pipeline("dataTI.xlsx")


FileNotFoundError: [Errno 2] No such file or directory: 'dataTI.xlsx'